In [2]:
%env WORKDIR=/tmp/vault
%env RESOURCE_GROUP=VaultDemoRG
%env APP_NAME=vaultsecretsync-mapfre
%env KEYVAULT=vaultsecretsync-mapfre
%env SUBSCRIPTION_ID=<azure-subscription-id>
%env TENANT_ID=<azure-tenant-id>

env: WORKDIR=/tmp/vault
env: RESOURCE_GROUP=VaultDemoRG
env: APP_NAME=vaultsecretsync-mapfre
env: KEYVAULT=vaultsecretsync-mapfre
env: SUBSCRIPTION_ID=<azure-subscription-id>
env: TENANT_ID=<azure-tenant-id>


Requires https://docs.prod.secops.hashicorp.services/doormat/azure/working_with_ad/

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE)

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')

# Log into Azure CLI

In [4]:
! az login --tenant $TENANT_ID --subscription $SUBSCRIPTION_ID

A web browser has been opened at https://login.microsoftonline.com/<azure-tenant-id>/oauth2/v2.0/authorize. Please continue the login in the web browser. If no web browser is available or if the web browser fails to open, use device code flow with `az login --use-device-code`.

Retrieving subscriptions for the selection...
[
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "<azure-tenant-id>",
    "id": "<azure-subscription-id>",
    "isDefault": true,
    "managedByTenants": [],
    "name": "secret-sync-mapfre-test",
    "state": "Enabled",
    "tenantId": "<azure-tenant-id>",
    "user": {
      "name": "jose.merchan@hashicorp.services",
      "type": "user"
    }
  }
]


In [5]:
%%bash
az account show \
  --query '{subscription:name, subscriptionId:id, tenantId:tenantId}' \
  --output table

Subscription             SubscriptionId                        TenantId
-----------------------  ------------------------------------  ------------------------------------
secret-sync-mapfre-test  <azure-subscription-id>  <azure-tenant-id>


In [6]:
%%bash
az provider register \
  --namespace Microsoft.KeyVault \
  --subscription "$SUBSCRIPTION_ID"

In [7]:
! az provider show \
  --namespace Microsoft.KeyVault \
  --subscription "$SUBSCRIPTION_ID" \
  --query '{namespace:namespace,state:registrationState}' \
  --output table

Namespace           State
------------------  ----------
Microsoft.KeyVault  Registered


### Create Resource Group

In [8]:
! az group create --name $RESOURCE_GROUP --location westeurope 

{
  "id": "/subscriptions/<azure-subscription-id>/resourceGroups/VaultDemoRG",
  "location": "westeurope",
  "managedBy": null,
  "name": "VaultDemoRG",
  "properties": {
    "provisioningState": "Succeeded"
  },
  "tags": null,
  "type": "Microsoft.Resources/resourceGroups"
}


In [9]:
! az group show --name $RESOURCE_GROUP | jq -r '.id'

/subscriptions/<azure-subscription-id>/resourceGroups/VaultDemoRG


## Create KeyVault

In [10]:
%%bash

az keyvault create --name $KEYVAULT --resource-group $RESOURCE_GROUP --location westeurope 

{
  "id": "/subscriptions/<azure-subscription-id>/resourceGroups/VaultDemoRG/providers/Microsoft.KeyVault/vaults/vaultsecretsync-mapfre",
  "location": "westeurope",
  "name": "vaultsecretsync-mapfre",
  "properties": {
    "accessPolicies": [],
    "createMode": null,
    "enablePurgeProtection": null,
    "enableRbacAuthorization": true,
    "enableSoftDelete": true,
    "enabledForDeployment": false,
    "enabledForDiskEncryption": false,
    "enabledForTemplateDeployment": false,
    "hsmPoolResourceId": null,
    "networkAcls": null,
    "privateEndpointConnections": null,
    "provisioningState": "Succeeded",
    "publicNetworkAccess": "Enabled",
    "sku": {
      "family": "A",
      "name": "standard"
    },
    "softDeleteRetentionInDays": 90,
    "tenantId": "<azure-tenant-id>",
    "vaultUri": "https://vaultsecretsync-mapfre.vault.azure.net/"
  },
  "resourceGroup": "VaultDemoRG",
  "systemData": {
    "createdAt": "2026-07-23T16:20:56.781000+00:00",
    "createdBy": "jose.

### Create App, SP, Role Assignment and Credentials

In [11]:
# to delete the app registration
# az ad app delete --id 100cb49d-f583-42b7-ab60-56f89dc0338b

In [12]:
%%bash
echo "Create App Registration"
export APP_ID=$(az ad app create --display-name $APP_NAME | jq -r .appId)
echo $APP_ID

echo "Create Service Principal"
az ad sp create --id $APP_ID

Create App Registration
b045247c-39e6-40f7-992b-65cdfae8835e
Create Service Principal
{
  "@odata.context": "https://graph.microsoft.com/v1.0/$metadata#servicePrincipals/$entity",
  "accountEnabled": true,
  "addIns": [],
  "alternativeNames": [],
  "appDescription": null,
  "appDisplayName": "vaultsecretsync-mapfre",
  "appId": "b045247c-39e6-40f7-992b-65cdfae8835e",
  "appOwnerOrganizationId": "<azure-tenant-id>",
  "appRoleAssignmentRequired": false,
  "appRoles": [],
  "applicationTemplateId": null,
  "createdByAppId": "04b07795-8ddb-461a-bbee-02f9e1bf7b46",
  "createdDateTime": "2026-07-23T16:42:54Z",
  "deletedDateTime": null,
  "description": null,
  "disabledByMicrosoftStatus": null,
  "displayName": "vaultsecretsync-mapfre",
  "homepage": null,
  "id": "a40e2ba2-27dd-47f6-84a2-5696e4732d6e",
  "info": {
    "logoUrl": null,
    "marketingUrl": null,
    "privacyStatementUrl": null,
    "supportUrl": null,
    "termsOfServiceUrl": null
  },
  "isDisabled": null,
  "keyCredentia

## Assign permisions to app

In [13]:
%%bash
CLIENT_ID=$(az ad app list --display-name $APP_NAME --query "[0].appId" -o tsv)
az role assignment create \
  --assignee $CLIENT_ID \
  --role "Key Vault Administrator" \
  --scope "/subscriptions/$SUBSCRIPTION_ID/resourceGroups/$RESOURCE_GROUP/providers/Microsoft.KeyVault/vaults/$KEYVAULT" || true

{
  "condition": null,
  "conditionVersion": null,
  "createdBy": null,
  "createdOn": "2026-07-23T16:42:59.634425+00:00",
  "delegatedManagedIdentityResourceId": null,
  "description": null,
  "id": "/subscriptions/<azure-subscription-id>/resourceGroups/VaultDemoRG/providers/Microsoft.KeyVault/vaults/vaultsecretsync-mapfre/providers/Microsoft.Authorization/roleAssignments/2ceda58e-408a-4ebf-94a2-a13c41876abf",
  "name": "2ceda58e-408a-4ebf-94a2-a13c41876abf",
  "principalId": "a40e2ba2-27dd-47f6-84a2-5696e4732d6e",
  "principalType": "ServicePrincipal",
  "resourceGroup": "VaultDemoRG",
  "roleDefinitionId": "/subscriptions/<azure-subscription-id>/providers/Microsoft.Authorization/roleDefinitions/00482a5a-887f-4fb3-b363-3b7fe8e74483",
  "scope": "/subscriptions/<azure-subscription-id>/resourceGroups/VaultDemoRG/providers/Microsoft.KeyVault/vaults/vaultsecretsync-mapfre",
  "systemData": null,
  "type": "Microsoft.Authorization/roleAssignments",
  "updatedBy": "0518caac-617d-4fe4-8cee-

## Configure Vault to connect

In [14]:
%%bash
set -euo pipefail

CLIENT_ID=$(
  az ad app list \
    --display-name "$APP_NAME" \
    --query "[0].appId" \
    --output tsv
)

CLIENT_SECRET=$(
  az ad app credential reset \
    --id "$CLIENT_ID" \
    --display-name "vault-secrets-sync" \
    --append \
    --years 1 \
    --query "password" \
    --output tsv \
    --only-show-errors |
  tr -d '\r'
)

VAULT_URI=$(
  az keyvault show \
    --name "$KEYVAULT" \
    --resource-group "$RESOURCE_GROUP" \
    --query "properties.vaultUri" \
    --output tsv
)

if [[ -z "$CLIENT_ID" || -z "$CLIENT_SECRET" || -z "$VAULT_URI" ]]; then
  echo "A required Azure value is empty" >&2
  exit 1
fi

sleep 50

vault write sys/sync/destinations/azure-kv/azure-sync \
  key_vault_uri="$VAULT_URI" \
  client_id="$CLIENT_ID" \
  client_secret="$CLIENT_SECRET" \
  tenant_id="$TENANT_ID" \
  secret_name_template='vault_sync_{{ .SecretBaseName | lowercase }}'


unset CLIENT_SECRET

Key                   Value
---                   -----
connection_details    map[client_id:b045247c-39e6-40f7-992b-65cdfae8835e client_secret:***** key_vault_uri:https://vaultsecretsync-mapfre.vault.azure.net/ tenant_id:<azure-tenant-id>]
name                  azure-sync
options               map[custom_tags:map[] granularity_level:secret-path secret_name_template:vault_sync_{{ .SecretBaseName | lowercase }}]
type                  azure-kv


## Add permissions to login user for access to the Key Vault

In [15]:
 %%bash
 az role assignment create \
  --assignee $(az ad signed-in-user show | jq -r .id) \
  --role "Key Vault Administrator" \
  --scope "/subscriptions/$SUBSCRIPTION_ID/resourceGroups/$RESOURCE_GROUP/providers/Microsoft.KeyVault/vaults/$KEYVAULT"

{
  "condition": null,
  "conditionVersion": null,
  "createdBy": null,
  "createdOn": "2026-07-23T16:44:12.455608+00:00",
  "delegatedManagedIdentityResourceId": null,
  "description": null,
  "id": "/subscriptions/<azure-subscription-id>/resourceGroups/VaultDemoRG/providers/Microsoft.KeyVault/vaults/vaultsecretsync-mapfre/providers/Microsoft.Authorization/roleAssignments/754ef202-432c-434f-b0fa-c1c76b9a90db",
  "name": "754ef202-432c-434f-b0fa-c1c76b9a90db",
  "principalId": "0518caac-617d-4fe4-8cee-2809b74be4c2",
  "principalType": "User",
  "resourceGroup": "VaultDemoRG",
  "roleDefinitionId": "/subscriptions/<azure-subscription-id>/providers/Microsoft.Authorization/roleDefinitions/00482a5a-887f-4fb3-b363-3b7fe8e74483",
  "scope": "/subscriptions/<azure-subscription-id>/resourceGroups/VaultDemoRG/providers/Microsoft.KeyVault/vaults/vaultsecretsync-mapfre",
  "systemData": null,
  "type": "Microsoft.Authorization/roleAssignments",
  "updatedBy": "0518caac-617d-4fe4-8cee-2809b74be4c2

In [16]:
%%bash
az keyvault secret list --vault-name $KEYVAULT --query "[].{Name:name, Value:value}" -o table

Name
---------------
vault-sync-test


# Read the sync secret

In [17]:
! # Read the Azure sync destination
! vault read sys/sync/destinations/azure-kv/azure-sync

Key                   Value
---                   -----
connection_details    map[client_id:b045247c-39e6-40f7-992b-65cdfae8835e client_secret:***** key_vault_uri:https://vaultsecretsync-mapfre.vault.azure.net/ tenant_id:<azure-tenant-id>]
name                  azure-sync
options               map[granularity_level:secret-path secret_name_template:vault_sync_{{ .SecretBaseName | lowercase }}]
type                  azure-kv


In [18]:
!# Read all associations and their sync status
!vault read sys/sync/destinations/azure-kv/azure-sync/associations

Key                          Value
---                          -----
associated_secrets           map[kv_49ccfaec/test:map[accessor:kv_49ccfaec external_name:vault-sync-test last_operation:Destroy last_sync_error_code:internal error mount:kv secret_name:test sync_status:INTERNAL_VAULT_ERROR updated_at:2026-07-23T16:41:56.938073461Z]]
store_name                   azure-sync
store_type                   azure-kv
sync_operation_counters      map[SYNCED:1]
unsync_operation_counters    map[INTERNAL_VAULT_ERROR:1]


In [19]:
%%bash
az keyvault secret show \
  --vault-name "$KEYVAULT" \
  --name "vault-sync-test" \
  --query '{name:name, enabled:attributes.enabled, updated:attributes.updated}' \
  --output json

{
  "enabled": true,
  "name": "vault-sync-test",
  "updated": "2026-07-23T16:26:49+00:00"
}


In [20]:
%%bash
az keyvault secret show \
  --vault-name "$KEYVAULT" \
  --name "vault-sync-test" \
  --query value \
  --output tsv

{"sync_retry":"2026-07-23T14:46:41Z","test":"test"}


# CLEAN UP

Run this only when you want to remove the PoC. Vault is cleaned first so it can unsync external secrets while the SPN and Key Vault still exist.

In [21]:
%%bash
set -euo pipefail

DESTINATION_PATH="sys/sync/destinations/azure-kv/azure-sync"

echo "Current Vault associations"
vault read -format=json "$DESTINATION_PATH/associations" | jq '.data.associated_secrets // {}' || true

echo "Purging the Vault destination and unsyncing associated secrets"
if vault read "$DESTINATION_PATH" >/dev/null 2>&1; then
  vault delete "$DESTINATION_PATH" purge=true

  destination_deleted=false
  for attempt in {1..24}; do
    if ! vault read "$DESTINATION_PATH" >/dev/null 2>&1; then
      destination_deleted=true
      echo "Vault destination deleted"
      break
    fi
    echo "Waiting for Vault to finish the purge (${attempt}/24)..."
    sleep 5
  done

  if [[ "$destination_deleted" != "true" ]]; then
    echo "Vault destination still exists. Azure resources were not deleted." >&2
    echo "Inspect the association status before considering force_delete." >&2
    exit 1
  fi
else
  echo "Vault destination does not exist; continuing"
fi

CLIENT_ID=$(az ad app list --display-name "$APP_NAME" --query '[0].appId' --output tsv)

if [[ -n "$CLIENT_ID" ]]; then
  echo "Deleting the service principal"
  az ad sp delete --id "$CLIENT_ID" || true

  echo "Deleting the app registration"
  az ad app delete --id "$CLIENT_ID" || true
else
  echo "App registration does not exist; continuing"
fi

if az keyvault show --name "$KEYVAULT" --resource-group "$RESOURCE_GROUP" >/dev/null 2>&1; then
  echo "Deleting Azure Key Vault"
  az keyvault delete --name "$KEYVAULT" --resource-group "$RESOURCE_GROUP"
fi

if az keyvault show-deleted --name "$KEYVAULT" >/dev/null 2>&1; then
  echo "Purging the soft-deleted Azure Key Vault"
  az keyvault purge --name "$KEYVAULT" --location westeurope
fi

if az group exists --name "$RESOURCE_GROUP" | grep -qx true; then
  echo "Deleting the resource group"
  az group delete --name "$RESOURCE_GROUP" --yes
fi

echo "Cleanup completed"

Current Vault associations


{}
Purging the Vault destination and unsyncing associated secrets
Success! Data deleted (if it existed) at: sys/sync/destinations/azure-kv/azure-sync
Vault destination deleted
Deleting the service principal
Deleting the app registration
Deleting Azure Key Vault


https://learn.microsoft.com/azure/key-vault/general/soft-delete-overview


Purging the soft-deleted Azure Key Vault
Deleting the resource group
Cleanup completed
